# Feature Engineering  

This notebook uses the cleaned data from the folder data/processed and creates additional features.  

In [64]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from ydata_profiling import ProfileReport
from c08_farming_exit import config, features, data_cleaning, mappings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Import processed data

In [66]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "clean_data.csv")

## 2. Add new features

### 2.1 Employment features

#### 2.1.1 Employment categories

In [60]:
#EMPLOYMENT CATEGORIES
conditions = [
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].isnull()),
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].notnull()),
    (df["farm_empl_last_12_months"] == 0) & (df["empl_type"].notnull()),
]
choices = ["only_farm", "hybrid", "fully_off_farm"]

df["empl_category"] = np.select(conditions, choices, default=None)

#### 2.1.2 Work hours per year

In [ ]:
#ABSOLUTE WORK HOURS PER YEAR
# farming
df["cash_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_cash_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_cash_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_cash_crops_days_per_week"] * 4)
    * df["farm_empl_cash_crops_hours_per_day"],
    np.nan
)

df["food_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_food_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_food_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_food_crops_days_per_week"] * 4)
    * df["farm_empl_food_crops_hours_per_day"],
    np.nan
)

df["livestock_hours_per_year"] = (
    (df["farm_empl_livestock_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_livestock_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_livestock_days_per_week"] * 4)
    * df["farm_empl_livestock_hours_per_day"]
)

df["farm_hours_per_year"] = df["cash_crop_hours_per_year"] + df["food_crop_hours_per_year"] + df["livestock_hours_per_year"]

# self-employment
df["self_empl_hours_per_year"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_duration_rainy_season_in_months_last_12_months"]
     + df["self_empl_duration_dry_season_in_months_last_12_months"])
    * (df["self_empl_days_per_week"] * 4)
    * df["self_empl_hours_per_day"],
    np.nan
)

# permanent wage employment
df["wage_empl_permanent_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Permanent",
    12
    * (df["wage_empl_permanent_days_per_week"] * 4)
    * df["wage_empl_permanent_hours_per_day"],
    np.nan
)

# seasonal/casual wage employment
df["wage_empl_seasonal_casual_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Seasonal",
    (df["wage_empl_seasonal_casual_rainy_season_duration_in_months_last_12_months"]
     + df["wage_empl_seasonal_casual_dry_season_duration_in_months_last_12_months"])
    * (df["wage_empl_seasonal_casual_duration_days_per_week"] * 4)
    * df["wage_empl_seasonal_casual_duration_hours_per_day"],
    np.nan
)

#total yearly work hours
cols = [
    "farm_hours_per_year",
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]
df["total_work_hours_per_year"] = df[cols].sum(axis=1, min_count=1)


In [ ]:
#RELATIVE WORK HOURS PER YEAR
# avoid dividing by zero -> treat a total of 0 hours as NaN (undefined share)
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)

df["farm_hours_share"] = df["farm_hours_per_year"] / total_safe
df["self_empl_hours_share"] = df["self_empl_hours_per_year"] / total_safe
df["wage_empl_permanent_hours_share"] = df["wage_empl_permanent_hours_per_year"] / total_safe
df["wage_empl_seasonal_casual_hours_share"] = df["wage_empl_seasonal_casual_hours_per_year"] / total_safe


In [ ]:
# OFF-FARM WORK SHARE
off_farm_cols = [
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]

# sum off-farm categories, treating "not applicable" (NaN) as 0 
df["off_farm_hours_per_year"] = df[off_farm_cols].sum(axis=1, min_count=0)

# but if the person has NO work data at all, keep it NaN rather than 0
df.loc[df["total_work_hours_per_year"].isna(), "off_farm_hours_per_year"] = np.nan

# share of total work time that is off-farm
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)
df["off_farm_share"] = df["off_farm_hours_per_year"] / total_safe

#### 2.1.1 Wage per hour

In [ ]:
#ABSOLUTE WAGES PER HOUR
# farming
##TODO: calculate it based on the casual wage employment for the sector agriculture per country

# self-employment
input_costs_cols = [
    "self_empl_input_costs_last_30_days",
    "self_empl_labor_costs_last_30_days",
    "self_empl_capital_costs_last_30_days",
]

# sum off-farm categories, treating "not applicable" (NaN) as 0 
df["self_empl_input_costs"] = df[input_costs_cols].sum(axis=1, min_count=0)

df["self_empl_wage_per_month"] = df["self_empl_sales_last_30_days"] - df["self_empl_input_costs"]

df["self_empl_hours_per_month"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_days_per_week"] * 4)
    * df["self_empl_hours_per_day"],
    np.nan
)

df["self_empl_wage_per_hour"] = df["self_empl_wage_per_month"] / df["self_empl_hours_per_month"]
                
# permanent wage employment
#TODO: calculate based on "wage_empl_permanent_wage_per_month" divided by work hours per months 

# seasonal/casual wage employment
#TODO: more complex calculation based on payment_frequency and wage_per_interval


#### 2.1.3 Employment Drop

In [ ]:
#DROPPING CALCULATION INPUTS
df = df.drop(columns=[
    # Farm employment - cash crops
    "farm_empl_cash_crops_duration_rainy_season_in_months_last_12_months",
    "farm_empl_cash_crops_duration_dry_season_in_months_last_months",
    "farm_empl_cash_crops_days_per_week",
    "farm_empl_cash_crops_hours_per_day",
    "cash_crop_hours_per_year",

    # Farm employment - food crops
    "farm_empl_food_crops_duration_rainy_season_in_months_last_12_months",
    "farm_empl_food_crops_duration_dry_season_in_months_last_12_months",
    "farm_empl_food_crops_days_per_week",
    "farm_empl_food_crops_hours_per_day",
    "food_crop_hours_per_year"
   
    # Livestock employment
    "farm_empl_livestock_duration_rainy_season_in_months_last_12_months"
    "farm_empl_livestock_duration_dry_season_in_months_last_12_months"
    "farm_empl_livestock_days_per_week"
    "farm_empl_livestock_hours_per_day",
    "livestock_hours_per_year"

    # Self-employment duration
    "self_empl_duration_rainy_season_in_months_last_12_months",
    "self_empl_duration_dry_season_in_months_last_12_months",
    "self_empl_days_per_week",
    "self_empl_hours_per_day",

    # Wage employment - permanent
    "wage_empl_permanent_days_per_week",
    "wage_empl_permanent_hours_per_day",

    # Wage employment - seasonal/casual
    "wage_empl_seasonal_casual_rainy_season_duration_in_months_last_12_months",
    "wage_empl_seasonal_casual_dry_season_duration_in_months_last_12_months",
    "wage_empl_seasonal_casual_duration_days_per_week",
    "wage_empl_seasonal_casual_duration_hours_per_day",

    # Self-employment costs
    "self_empl_input_costs_last_30_days",
    "self_empl_labor_costs_last_30_days",
    "self_empl_capital_costs_last_30_days",

    # Misc self-employment columns
    "input_costs",
    "self_empl_input_costs",
    "self_empl_wage_per_month",
    "self_empl_hours_per_month"

    ])
